# Organizational Network Analysis (ONA)

This notebook demonstrates the ONA module of the People Analytics suite.

**What we cover:**
1. Build a collaboration network from interaction metadata
2. Visualize inter-department communication patterns
3. Detect communities and identify organizational silos
4. Map influence and find key connectors
5. Identify bottlenecks in information flow

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['figure.figsize'] = (10, 5)

from src.ona.network_builder import NetworkBuilder
from src.ona.community_detector import CommunityDetector
from src.ona.influence_mapper import InfluenceMapper

## 1. Load Collaboration Data

In [ ]:
collab = pd.read_csv('../data/collaboration_data.csv')
print(f'Interactions: {len(collab):,}')
print(f'Interaction types: {collab["interaction_type"].value_counts().to_dict()}')
print(f'Cross-department: {collab["is_cross_department"].mean():.1%}')
collab.head()

In [ ]:
# Interaction volume by type and department
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

collab['interaction_type'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Interactions by Type')
axes[0].set_ylabel('Count')

collab['sender_dept'].value_counts().plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Outgoing Interactions by Department')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 2. Build Collaboration Network

In [ ]:
builder = NetworkBuilder(min_interactions=3)
network = builder.build_department_network(collab)
metrics = builder.get_network_metrics(collab)

print(f'Nodes: {metrics.num_nodes}')
print(f'Edges: {metrics.num_edges}')
print(f'Density: {metrics.density:.3f}')
print(f'Avg degree: {metrics.avg_degree:.1f}')
print(f'Cross-dept %: {metrics.cross_dept_pct:.1%}')

In [ ]:
# Adjacency matrix heatmap
adj = network['adjacency_matrix']
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(adj.values, cmap='YlOrRd')
ax.set_xticks(range(len(adj.columns)))
ax.set_xticklabels(adj.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(adj.index)))
ax.set_yticklabels(adj.index, fontsize=9)
plt.colorbar(im, label='Interaction Count')
ax.set_title('Department Collaboration Matrix')

# Annotate cells
for i in range(len(adj.index)):
    for j in range(len(adj.columns)):
        val = adj.values[i, j]
        if val > 0:
            ax.text(j, i, f'{int(val)}', ha='center', va='center', fontsize=7,
                   color='white' if val > adj.values.max() * 0.6 else 'black')
plt.tight_layout()
plt.show()

In [ ]:
# Node metrics
node_metrics = builder.compute_node_metrics(collab)
display(node_metrics)

In [ ]:
# Internal vs external interaction ratio
fig, ax = plt.subplots(figsize=(10, 4))
nm = node_metrics.sort_values('internal_ratio', ascending=True)
ax.barh(nm['department'], nm['internal_ratio'], color='steelblue', label='Internal ratio')
ax.barh(nm['department'], 1 - nm['internal_ratio'], left=nm['internal_ratio'],
        color='#e74c3c', alpha=0.6, label='External ratio')
ax.set_xlabel('Proportion')
ax.set_title('Internal vs. Cross-Department Communication Ratio')
ax.legend()
ax.axvline(0.5, color='black', linestyle=':', linewidth=0.5)
plt.tight_layout()
plt.show()

## 3. Community Detection & Silo Identification

In [ ]:
detector = CommunityDetector(n_communities=3, silo_threshold=0.7)
communities = detector.detect(adj)

silo_report = detector.silo_report()
display(silo_report)

In [ ]:
# Network graph visualization using spring layout
fig, ax = plt.subplots(figsize=(10, 8))

# Simple force-directed layout
np.random.seed(42)
nodes = adj.index.tolist()
n = len(nodes)
pos = np.random.randn(n, 2)

# Iterate spring layout
matrix = adj.values.copy()
np.fill_diagonal(matrix, 0)
for _ in range(50):
    for i in range(n):
        force = np.zeros(2)
        for j in range(n):
            if i == j:
                continue
            diff = pos[j] - pos[i]
            dist = max(np.linalg.norm(diff), 0.01)
            # Repulsion
            force -= diff / (dist ** 2) * 0.1
            # Attraction (weighted)
            if matrix[i, j] > 0:
                force += diff * matrix[i, j] / matrix.max() * 0.01
        pos[i] += force * 0.1

# Assign colors by community
community_colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
membership = detector.community_membership()
node_colors = []
for node in nodes:
    cid = membership[membership['node'] == node]['community_id'].values[0]
    node_colors.append(community_colors[cid % len(community_colors)])

# Draw edges
for i in range(n):
    for j in range(i + 1, n):
        if matrix[i, j] > 0:
            weight = matrix[i, j] / matrix.max()
            ax.plot([pos[i, 0], pos[j, 0]], [pos[i, 1], pos[j, 1]],
                   'gray', alpha=min(weight * 2, 0.6), linewidth=weight * 3)

# Draw nodes
node_sizes = node_metrics.set_index('department').loc[nodes, 'total_interactions'].values
node_sizes = (node_sizes / node_sizes.max()) * 800 + 200
for i, node in enumerate(nodes):
    ax.scatter(pos[i, 0], pos[i, 1], s=node_sizes[i], c=node_colors[i],
              edgecolors='white', linewidth=2, zorder=3)
    ax.annotate(node, (pos[i, 0], pos[i, 1]), fontsize=8, ha='center', va='bottom',
               fontweight='bold', xytext=(0, 10), textcoords='offset points')

# Legend for communities
for cid in range(len(communities)):
    silo_label = ' (SILO)' if communities[cid].is_silo else ''
    ax.scatter([], [], c=community_colors[cid], s=100,
              label=f'Community {cid}: {", ".join(communities[cid].members[:3])}{silo_label}')
ax.legend(fontsize=8, loc='upper left')
ax.set_title('Organizational Collaboration Network (colored by community)')
ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Influence Mapping & Key Connectors

In [ ]:
mapper = InfluenceMapper()
influence = mapper.influence_table(adj)
display(influence)

In [ ]:
# Centrality comparison radar chart
fig, ax = plt.subplots(figsize=(10, 5))
metrics_cols = ['degree_centrality', 'betweenness_centrality', 'eigenvector_centrality']
x = np.arange(len(influence))
width = 0.25
for i, col in enumerate(metrics_cols):
    ax.bar(x + i * width, influence[col], width, label=col.replace('_', ' ').title())
ax.set_xticks(x + width)
ax.set_xticklabels(influence['department'], rotation=45, ha='right')
ax.set_ylabel('Centrality Score')
ax.set_title('Network Centrality Metrics by Department')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Influence score with role annotations
fig, ax = plt.subplots(figsize=(10, 4))
inf_sorted = influence.sort_values('influence_score')
role_colors = {'connector': '#2ecc71', 'hub': '#3498db', 'bottleneck': '#e74c3c', 'peripheral': '#95a5a6'}
bar_colors = [role_colors.get(r, '#95a5a6') for r in inf_sorted['role']]
ax.barh(inf_sorted['department'], inf_sorted['influence_score'], color=bar_colors)
ax.set_xlabel('Composite Influence Score')
ax.set_title('Department Influence Scores (colored by network role)')
for i, (_, row) in enumerate(inf_sorted.iterrows()):
    ax.text(row['influence_score'] + 0.005, i, row['role'].upper(),
            va='center', fontsize=8, color=role_colors.get(row['role'], 'gray'))
plt.tight_layout()
plt.show()

In [ ]:
# Key connectors and bottlenecks
connectors = mapper.find_connectors(adj, top_n=3)
bottlenecks = mapper.find_bottlenecks(adj)

print('Top Cross-Department Connectors:')
display(connectors[['department', 'betweenness_centrality', 'degree_centrality', 'role']])

if not bottlenecks.empty:
    print('\nPotential Bottlenecks (high betweenness, low degree):')
    display(bottlenecks[['department', 'betweenness_centrality', 'degree_centrality', 'role']])
else:
    print('\nNo bottlenecks detected.')

## Key Takeaways

- **Adjacency matrix** reveals which departments collaborate heavily vs. operate in isolation
- **Community detection** surfaces natural clusters — silos indicate teams that need cross-functional intervention
- **Influence mapping** identifies informal leaders (connectors) and single points of failure (bottlenecks)
- **Production path:** integrate with Microsoft Graph API for real email/Teams/calendar metadata; scale with graph databases (Neo4j)